# UI-Flow Haptic Pipeline v3

This notebook converts a three-image UI flow into a 2-second haptic waveform with an event timeline mixer.

v3 keeps the eight atlas classes as a coarse router, but the LLM now returns 1-3 ordered semantic events. Each event chooses a behavior class and a curated prototype index from the prototype catalog. Code, not the LLM, assigns deterministic timeline positions so repeated generations of the same plan do not drift in time.

Outputs follow the HapticGen UI validation artifact layout and add an `events/` folder with per-event waveforms and metadata.


In [ ]:
# 1. Clone code repo (or force-update to latest)
import os
import subprocess
import sys

REPO_URL = "https://github.com/cindy-77jiayi/thesis_hapticAE.git"
REPO_DIR = "/content/thesis_hapticAE"
BRANCH = 'altas'
FORCE_CLEAN_CLONE = False

if FORCE_CLEAN_CLONE and os.path.exists(REPO_DIR):
    subprocess.run(["rm", "-rf", REPO_DIR], check=True)

if not os.path.exists(os.path.join(REPO_DIR, ".git")):
    clone_cmd = ["git", "clone", REPO_URL, REPO_DIR]
    if BRANCH:
        clone_cmd = ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_DIR]
    subprocess.run(clone_cmd, check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--all"], check=False)
    if BRANCH:
        subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH], check=False)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f"Repo ready at: {REPO_DIR}")

In [ ]:
# -----------------------------
# Editable notebook parameters
# -----------------------------
from pathlib import Path

CONFIG_PATH = "configs/vae_balanced_8d_0p5s.yaml"
CHECKPOINT_PATH_OVERRIDE = "/content/drive/MyDrive/thesis_outputs/outputs/vae_balanced_8d_0p5s/best_model.pt"
REPO_URL = "https://github.com/cindy-77jiayi/thesis_hapticAE.git"
REPO_BRANCH = "altas"
REPO_DIR = Path("/content/thesis_hapticAE")

DRIVE_ROOT = Path("/content/drive/MyDrive/thesis_outputs")
OUTPUTS_ROOT = DRIVE_ROOT / "outputs"
ATLAS_DIR = DRIVE_ROOT / "behavior_atlas_8class"
CURATED_JSON_PATH = ATLAS_DIR / "curated_behavior_prototypes.json"
ATLAS_JSON_PATH = ATLAS_DIR / "behavior_atlas.json"
LATENT_TABLE_PATH = ATLAS_DIR / "latent_behavior_table.csv"
PIPELINE_OUTPUT_DIR = DRIVE_ROOT / "ui_flow_haptic_v3"

BEHAVIOR_CLASSES = [
    "quiet_or_subtle",
    "single_short_pulse",
    "double_short_pulse",
    "multi_short_pulse",
    "long_sustained",
    "long_rising",
    "long_falling",
    "long_modulated_or_pulsed",
]
SEMANTIC_ROLE_OPTIONS = [
    "press_down",
    "drag_motion",
    "pull_motion",
    "scroll_motion",
    "release",
    "confirmation",
    "notification",
    "warning",
    "state_change",
    "subtle_context",
]
STYLE_OPTIONS = [
    "soft",
    "sharp",
    "strong",
    "smooth",
    "ticked",
    "bumpy",
    "rising",
    "falling",
    "sustained",
    "subtle",
]
TIMING_HINT_OPTIONS = ["beginning", "middle", "end"]
DURATION_HINT_OPTIONS = ["short", "medium", "long", "full"]
DURATION_HINT_SECONDS = {
    "short": 0.5,
    "medium": 0.8,
    "long": 1.2,
    "full": 2.0,
}

# v3 uses the LLM path directly. Fill the three image paths before running.
USE_LLM_PLAN = True
OPENAI_MODEL = "gpt-4.1-mini"

# Randomized LLM-to-VAE input generation. Each run asks the LLM to produce
# a random schema-valid VAE input plan: random class/prototype/style/strength/
# duration/event count. The final 2s uint8 playback sequence is still derived
# from VAE-decoded waveform energy, not from direct random uint8 noise.
LLM_RANDOMIZE_OUTPUT = True
LLM_RANDOM_VAE_INPUT_IGNORE_UI = True
LLM_TEMPERATURE = 1.1
LLM_TOP_P = 1.0
LLM_RANDOMIZATION_SALT = None

BEFORE_IMAGE_PATH = None   # e.g. DRIVE_ROOT / "ui_flow_examples" / "before.png"
DURING_IMAGE_PATH = None   # e.g. DRIVE_ROOT / "ui_flow_examples" / "during.png"
AFTER_IMAGE_PATH = None    # e.g. DRIVE_ROOT / "ui_flow_examples" / "after.png"

# HAPTIC_PLAN is produced by the LLM cell below.
HAPTIC_PLAN = None
PLAN_SOURCE = "llm"
LLM_PLAN_RESULT = None

RANDOM_SEQUENCE_JSON_NAME = "random_2s_uint8_sequence.json"
FINAL_DURATION_S = 2.0
FADE_MS = 10.0
EVENT_GAP_S = 0.08
MIN_EVENT_GAP_S = 0.02
MIN_EVENT_DURATION_S = 0.35
TIMELINE_END_ALIGN_S = 1.9
MAX_EVENTS = 3
DEFAULT_DURATION_MODE = "hold_stretch"
MAX_CATALOG_PROTOTYPES_PER_CLASS = 20
CATALOG_FEATURE_KEYS = [
    "event_count",
    "active_ratio",
    "main_duration_s",
    "rms_energy",
    "crest_factor",
    "am_modulation_index",
    "envelope_slope",
    "envelope_decay_slope_dBps",
    "spectral_centroid_hz",
]


## LLM Contract

The LLM receives three screenshots and a compact curated prototype catalog. It must return one JSON object with 1-3 ordered events.

Each event chooses:

- `behavior_class`: one of the eight atlas classes.
- `semantic_role`: one of the fixed semantic roles.
- `style`: one of the fixed style words.
- `prototype_index`: an index inside that class's curated catalog, not a global sample index.
- `timing_hint` and `duration_hint`: soft hints only. Code assigns deterministic `start_idx` and `end_idx`.

The code path is:

`screenshots -> LLM event plan -> class/prototype routing -> raw-z decode -> duration stretch -> fade -> deterministic timeline mix -> HapticGen-style artifacts`

## Semantic-to-Z Learning Roadmap

v3 is still `semantic -> class/prototype choice -> raw z`, not direct `semantic -> z`.

The next learning step is:

1. Build a training table from atlas waveforms: raw z, waveform features, and coarse class.
2. Add semantic labels from curated prototypes and future UI-flow LLM event outputs: role, style, event type, timing, and strength.
3. Define a semantic vector: class one-hot, role/style one-hot, strength, duration, timing, risingness, modulation, sharpness, and event count.
4. Start with kNN or prototype interpolation inside each class.
5. Then train a small mapper `semantic vector -> raw z` while freezing the VAE.
6. Use `z` regression plus decoded waveform feature loss plus event-aware perceptual loss.
7. Validate by decoding predicted z and checking event category, strength, duration, modulation, and subjective haptic fit.


In [ ]:
# -----------------------------
# Setup, Drive mount, imports
# -----------------------------
import base64
import csv
import json
import mimetypes
import os
import shutil
import subprocess
import sys
import zipfile
from datetime import datetime
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Audio, display
from scipy.io.wavfile import write as wav_write

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print(f"Google Drive mount skipped or unavailable: {exc}")


def resolve_project_root() -> Path:
    current = Path.cwd()
    for candidate in [current, *current.parents]:
        if (candidate / "src").exists():
            return candidate

    if not (REPO_DIR / ".git").exists():
        print(f"Repository not found at {REPO_DIR}; cloning {REPO_BRANCH}...")
        subprocess.run(
            ["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)],
            check=True,
        )
    else:
        print(f"Repository found at {REPO_DIR}; updating {REPO_BRANCH}...")
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH], check=False)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=False)
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_BRANCH], check=False)

    if not (REPO_DIR / "src").exists():
        raise FileNotFoundError(f"Could not find src/ in current directory or cloned repo: {REPO_DIR}")
    return REPO_DIR


PROJECT_ROOT = resolve_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.loaders import build_model, load_checkpoint
from src.utils.config import load_config

PIPELINE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")
print(f"Pipeline output dir: {PIPELINE_OUTPUT_DIR}")


In [ ]:
# -----------------------------
# File resolution and prototype catalog helpers
# -----------------------------
def resolve_checkpoint(outputs_root: Path, override: str | None = None) -> Path:
    if override:
        path = Path(override)
        if not path.exists():
            raise FileNotFoundError(f"CHECKPOINT_PATH_OVERRIDE does not exist: {path}")
        print(f"Using override checkpoint: {path}")
        return path

    if not outputs_root.exists():
        raise FileNotFoundError(f"OUTPUTS_ROOT does not exist: {outputs_root}")

    candidates = []
    for name in ("best_model.pt", "last_checkpoint.pt"):
        candidates.extend(outputs_root.rglob(name))
    candidates = sorted(candidates, key=lambda p: p.stat().st_mtime, reverse=True)
    if not candidates:
        raise FileNotFoundError(f"No best_model.pt or last_checkpoint.pt found under: {outputs_root}")

    print("Checkpoint candidates, newest first:")
    for idx, path in enumerate(candidates[:10]):
        print(f"  [{idx}] {path}")
    print(f"Selected checkpoint: {candidates[0]}")
    return candidates[0]


def load_json(path: Path) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def load_prototype_source(curated_path: Path, atlas_path: Path) -> tuple[dict | None, dict | None]:
    curated = load_json(curated_path) if curated_path.exists() else None
    atlas = load_json(atlas_path) if atlas_path.exists() else None
    if curated is None and atlas is None:
        raise FileNotFoundError(
            f"Missing both curated prototype file and atlas file:\n  {curated_path}\n  {atlas_path}"
        )
    if curated is not None:
        print(f"Loaded curated prototypes: {curated_path}")
    else:
        print(f"Curated prototypes not found, using atlas only: {atlas_path}")
    if atlas is not None:
        print(f"Loaded atlas: {atlas_path}")
    return curated, atlas


def load_latent_feature_lookup(path: Path) -> dict[int, dict]:
    if not path.exists():
        print(f"latent_behavior_table.csv not found: {path}")
        return {}
    lookup: dict[int, dict] = {}
    with open(path, "r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if not row.get("sample_index"):
                continue
            idx = int(float(row["sample_index"]))
            converted = {}
            for key, value in row.items():
                if value is None:
                    converted[key] = value
                    continue
                try:
                    converted[key] = float(value)
                except (TypeError, ValueError):
                    converted[key] = value
            lookup[idx] = converted
    print(f"Loaded latent feature rows: {len(lookup)} from {path}")
    return lookup


def roles_for_sample(curated: dict | None, behavior_class: str, sample_index: int | None) -> list[str]:
    if curated is None or sample_index is None:
        return []
    class_entry = curated.get("classes", {}).get(behavior_class, {})
    roles = []
    for role_name, role_indices in (class_entry.get("roles") or {}).items():
        if int(sample_index) in {int(x) for x in role_indices}:
            roles.append(str(role_name))
    return roles


def compact_features_for_sample(sample_index: int | None, feature_lookup: dict[int, dict]) -> dict:
    if sample_index is None or int(sample_index) not in feature_lookup:
        return {}
    row = feature_lookup[int(sample_index)]
    compact = {}
    for key in CATALOG_FEATURE_KEYS:
        if key in row and row[key] not in ("", None):
            value = row[key]
            compact[key] = round(float(value), 4) if isinstance(value, (int, float)) else value
    return compact


def prototype_candidates_for_class(
    behavior_class: str,
    curated: dict | None,
    atlas: dict | None,
    feature_lookup: dict[int, dict],
) -> list[dict]:
    candidates: list[dict] = []

    if curated is not None:
        class_entry = curated.get("classes", {}).get(behavior_class, {})
        for proto in class_entry.get("approved_prototypes", []) or []:
            item = dict(proto)
            item["selection_source"] = "curated.approved_prototypes"
            candidates.append(item)
        fallback = class_entry.get("fallback_prototype")
        if not candidates and fallback:
            item = dict(fallback)
            item["selection_source"] = "curated.fallback_prototype"
            candidates.append(item)

    if not candidates and atlas is not None:
        prototypes = atlas.get("classes", {}).get(behavior_class, {}).get("prototype_z", []) or []
        for proto in prototypes[:MAX_CATALOG_PROTOTYPES_PER_CLASS]:
            item = dict(proto)
            item["selection_source"] = "atlas.prototype_z"
            candidates.append(item)

    normalized = []
    for idx, item in enumerate(candidates[:MAX_CATALOG_PROTOTYPES_PER_CLASS]):
        sample_index = item.get("sample_index")
        sample_index = int(sample_index) if sample_index is not None else None
        entry = {
            "prototype_index": idx,
            "behavior_class": behavior_class,
            "sample_index": sample_index,
            "file_path": item.get("file_path"),
            "atlas_class": item.get("atlas_class") or item.get("behavior_class") or behavior_class,
            "selection_source": item.get("selection_source"),
            "roles": roles_for_sample(curated, behavior_class, sample_index),
            "features": compact_features_for_sample(sample_index, feature_lookup),
            "z": [float(v) for v in item.get("z", [])],
        }
        normalized.append(entry)
    return normalized


def build_prototype_catalog(
    curated: dict | None,
    atlas: dict | None,
    feature_lookup: dict[int, dict],
) -> tuple[dict[str, list[dict]], dict, str]:
    catalog = {
        cls: prototype_candidates_for_class(cls, curated, atlas, feature_lookup)
        for cls in BEHAVIOR_CLASSES
    }
    compact = {}
    text_lines = []
    for cls in BEHAVIOR_CLASSES:
        compact[cls] = []
        text_lines.append(f"{cls}:")
        if not catalog[cls]:
            text_lines.append("  no curated prototypes available; choose only for silence/subtle fallback")
            continue
        for item in catalog[cls]:
            compact_item = {
                "prototype_index": item["prototype_index"],
                "sample_index": item["sample_index"],
                "roles": item["roles"],
                "features": item["features"],
                "file_path": item["file_path"],
            }
            compact[cls].append(compact_item)
            role_text = ",".join(item["roles"]) if item["roles"] else "none"
            feature_text = ", ".join(f"{k}={v}" for k, v in item["features"].items()) or "no_features"
            text_lines.append(
                f"  [{item['prototype_index']}] sample_index={item['sample_index']} "
                f"roles={role_text} features=({feature_text}) file={item['file_path']}"
            )
    return catalog, compact, "\n".join(text_lines)


def _coerce_path(path_like, label: str) -> Path:
    if path_like is None or str(path_like).strip() == "":
        raise ValueError(f"{label} is empty. Set BEFORE_IMAGE_PATH, DURING_IMAGE_PATH, and AFTER_IMAGE_PATH.")
    path = Path(path_like)
    if not path.exists():
        raise FileNotFoundError(f"{label} does not exist: {path}")
    return path


def image_to_data_url(path_like, label: str) -> str:
    path = _coerce_path(path_like, label)
    mime_type = mimetypes.guess_type(str(path))[0] or "image/png"
    encoded = base64.b64encode(path.read_bytes()).decode("ascii")
    return f"data:{mime_type};base64,{encoded}"


In [ ]:
# -----------------------------
# Load VAE, checkpoint, atlas files, and prototype catalog
# -----------------------------
config = load_config(str(PROJECT_ROOT / CONFIG_PATH))
sr = int(config["data"]["sr"])
chunk_len = int(config["data"]["T"])
latent_dim = int(config["model"]["latent_dim"])
final_len = int(round(FINAL_DURATION_S * sr))
clip_range = config.get("loss", {}).get("clamp_range", None)
data_clip_range = config.get("data", {}).get("clip_range", None)
if clip_range is None and data_clip_range is not None:
    clip_abs = max(abs(float(data_clip_range[0])), abs(float(data_clip_range[1])))
else:
    clip_abs = float(clip_range or 5.0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint_path = resolve_checkpoint(OUTPUTS_ROOT, CHECKPOINT_PATH_OVERRIDE)
curated_data, atlas_data = load_prototype_source(CURATED_JSON_PATH, ATLAS_JSON_PATH)
latent_feature_lookup = load_latent_feature_lookup(LATENT_TABLE_PATH)

PROTOTYPE_CATALOG, PROTOTYPE_CATALOG_FOR_LLM, PROTOTYPE_CATALOG_TEXT = build_prototype_catalog(
    curated_data,
    atlas_data,
    latent_feature_lookup,
)

model = build_model(config, device)
load_checkpoint(model, str(checkpoint_path), device)
model.eval()

print(f"Device: {device}")
print(f"sr={sr}, chunk_len={chunk_len}, final_len={final_len}, latent_dim={latent_dim}")
print(f"clip_abs={clip_abs}")
print("Prototype catalog:")
print(PROTOTYPE_CATALOG_TEXT[:5000])


In [ ]:
# -----------------------------
# v3: LLM parses before/during/after screenshots into an event plan
# -----------------------------
HAPTIC_PLAN_RESPONSE_FORMAT = {
    "type": "json_schema",
    "json_schema": {
        "name": "ui_flow_haptic_event_plan",
        "strict": True,
        "schema": {
            "type": "object",
            "additionalProperties": False,
            "properties": {
                "interaction_type": {"type": "string"},
                "ui_change_summary": {"type": "string"},
                "confidence": {"type": "number", "minimum": 0.0, "maximum": 1.0},
                "events": {
                    "type": "array",
                    "minItems": 1,
                    "maxItems": MAX_EVENTS,
                    "items": {
                        "type": "object",
                        "additionalProperties": False,
                        "properties": {
                            "event_id": {"type": "string"},
                            "event_type": {"type": "string"},
                            "semantic_role": {"type": "string", "enum": SEMANTIC_ROLE_OPTIONS},
                            "behavior_class": {"type": "string", "enum": BEHAVIOR_CLASSES},
                            "style": {"type": "string", "enum": STYLE_OPTIONS},
                            "strength": {"type": "number", "minimum": 0.0, "maximum": 1.0},
                            "timing_hint": {"type": "string", "enum": TIMING_HINT_OPTIONS},
                            "duration_hint": {"type": "string", "enum": DURATION_HINT_OPTIONS},
                            "prototype_policy": {"type": "string", "enum": ["catalog_choice"]},
                            "prototype_index": {"type": "integer", "minimum": 0, "maximum": 19},
                            "reason": {"type": "string"},
                        },
                        "required": [
                            "event_id",
                            "event_type",
                            "semantic_role",
                            "behavior_class",
                            "style",
                            "strength",
                            "timing_hint",
                            "duration_hint",
                            "prototype_policy",
                            "prototype_index",
                            "reason",
                        ],
                    },
                },
                "composition_notes": {"type": "string"},
            },
            "required": [
                "interaction_type",
                "ui_change_summary",
                "confidence",
                "events",
                "composition_notes",
            ],
        },
    },
}

LLM_SYSTEM_PROMPT = """
You are a haptic interaction planner. You inspect a UI flow shown as three screenshots: before, during, and after.
Return one JSON object matching the provided schema. Do not include markdown or extra prose.
You may output 1 to 3 ordered haptic events. If the visual flow contains more than 3 moments, merge them into the 3 most perceptually important events.
You must choose prototype_index values from the provided class catalog. prototype_index is local to the selected behavior_class.
Do not invent start times. Code will assign deterministic timeline positions from event order, timing_hint, and duration_hint.
When randomization mode is enabled, choose one plausible variation instead of always choosing the safest/default prototype.
""".strip()

LLM_USER_PROMPT = f"""
Analyze the three screenshots in order: BEFORE, DURING, AFTER.
Describe the UI change as ordered haptic events, such as drag motion followed by confirmation, pull motion followed by notification, or press followed by state change.

Behavior class guidance:
- quiet_or_subtle: tiny or ambiguous change, disabled/no-op, no meaningful tactile event.
- single_short_pulse: tap, click, button press, one discrete selection or confirmation.
- double_short_pulse: two-step confirmation, toggle with clear before/after snap, paired cue.
- multi_short_pulse: warning, repeated notification, several brief impacts.
- long_sustained: steady drag/hold/slider motion with fairly even intensity.
- long_rising: progressive increase, pull/drag/slider moving toward stronger state, charging, expanding, gaining intensity.
- long_falling: release, collapse, easing out, dismiss, settling, decreasing intensity.
- long_modulated_or_pulsed: scrolling, scrub, repeated ticks during continuous motion, segmented progress.

Timing hint guidance:
- beginning: immediate press/down/start feedback.
- middle: continuous movement or hold feedback.
- end: release, completion, drop, settle, final confirmation.

Duration hint guidance:
- short: brief tap/click/pulse; code maps to 0.5s.
- medium: mid-length transition; code maps to 0.8s.
- long: sustained movement; code maps to 1.2s.
- full: whole-window continuous movement; code maps to 2.0s.

Valid behavior classes: {BEHAVIOR_CLASSES}
Valid semantic roles: {SEMANTIC_ROLE_OPTIONS}
Valid styles: {STYLE_OPTIONS}
Valid timing hints: {TIMING_HINT_OPTIONS}
Valid duration hints: {DURATION_HINT_OPTIONS}

Curated prototype catalog:
{PROTOTYPE_CATALOG_TEXT}
""".strip()


def build_llm_randomization_context() -> dict:
    if not LLM_RANDOMIZE_OUTPUT:
        return {
            "enabled": False,
            "salt": None,
            "instruction": "Randomization disabled; choose the most representative valid haptic plan.",
        }
    salt = LLM_RANDOMIZATION_SALT
    if salt is None or str(salt).strip() == "":
        salt = f"{datetime.now().isoformat()}_{int(np.random.randint(0, 2**31 - 1))}"
    if LLM_RANDOM_VAE_INPUT_IGNORE_UI:
        instruction = f"""
Randomization mode is enabled. Variation salt: {salt}.
Generate a random schema-valid VAE input plan. It does not need to be event-like, reasonable, or semantically consistent with the UI screenshots.
Randomly vary event count, behavior_class, semantic_role, style, strength, timing_hint, duration_hint, and prototype_index while staying inside the provided JSON schema and catalog constraints.
The final playback sequence must be produced by decoding the selected raw-z prototypes through the VAE and converting the waveform envelope to 10 ms uint8 bins.
""".strip()
    else:
        instruction = f"""
Randomization mode is enabled. Variation salt: {salt}.
Choose one plausible haptic interpretation for this UI flow, not necessarily the safest default.
When several prototypes fit the same class, vary prototype_index, style, strength, duration_hint, and event count within what the screenshots can justify.
Do not choose prototype_index=0 by habit; choose it only when it is the best variant.
Keep the event sequence semantically consistent with the three screenshots.
""".strip()
    return {
        "enabled": True,
        "salt": str(salt),
        "instruction": instruction,
    }


def ensure_openai_client():
    try:
        from google.colab import userdata
        key = userdata.get("OPENAI_API_KEY")
        if key and not os.environ.get("OPENAI_API_KEY"):
            os.environ["OPENAI_API_KEY"] = key
    except Exception:
        pass

    if not os.environ.get("OPENAI_API_KEY"):
        raise RuntimeError(
            "OPENAI_API_KEY is not set. In Colab, add it to Secrets or run "
            "os.environ['OPENAI_API_KEY']='...' before USE_LLM_PLAN=True."
        )

    try:
        from openai import OpenAI
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai"], check=True)
        from openai import OpenAI
    return OpenAI()


def normalize_event_plan(plan: dict) -> dict:
    raw_events = list(plan.get("events", []))
    if not raw_events and "behavior" in plan:
        behavior = dict(plan.get("behavior", {}))
        raw_events = [{
            "event_id": "event_01",
            "event_type": plan.get("interaction_type", "ui_flow"),
            "semantic_role": "subtle_context",
            "behavior_class": behavior.get("class", "quiet_or_subtle"),
            "style": "subtle",
            "strength": behavior.get("strength", 0.5),
            "timing_hint": behavior.get("placement", "middle"),
            "duration_hint": "short",
            "prototype_policy": "catalog_choice",
            "prototype_index": behavior.get("prototype_index", 0),
            "reason": behavior.get("reason", ""),
        }]
    if not raw_events:
        raise ValueError("HAPTIC_PLAN must contain at least one event")

    normalized_events = []
    for idx, event in enumerate(raw_events[:MAX_EVENTS], start=1):
        behavior_class = str(event.get("behavior_class", "quiet_or_subtle"))
        if behavior_class not in BEHAVIOR_CLASSES:
            behavior_class = "quiet_or_subtle"
        semantic_role = str(event.get("semantic_role", "subtle_context"))
        if semantic_role not in SEMANTIC_ROLE_OPTIONS:
            semantic_role = "subtle_context"
        style = str(event.get("style", "subtle"))
        if style not in STYLE_OPTIONS:
            style = "subtle"
        timing_hint = str(event.get("timing_hint", "middle"))
        if timing_hint not in TIMING_HINT_OPTIONS:
            timing_hint = "middle"
        duration_hint = str(event.get("duration_hint", "short"))
        if duration_hint not in DURATION_HINT_OPTIONS:
            duration_hint = "short"

        normalized_events.append({
            "event_id": f"event_{idx:02d}",
            "llm_event_id": str(event.get("event_id") or f"event_{idx:02d}"),
            "event_type": str(event.get("event_type", "ui_event")),
            "semantic_role": semantic_role,
            "behavior_class": behavior_class,
            "style": style,
            "strength": float(np.clip(float(event.get("strength", 0.6)), 0.0, 1.0)),
            "timing_hint": timing_hint,
            "duration_hint": duration_hint,
            "duration_s": float(DURATION_HINT_SECONDS[duration_hint]),
            "prototype_policy": "catalog_choice",
            "prototype_index": max(0, int(event.get("prototype_index", 0))),
            "reason": str(event.get("reason", "")),
        })

    return {
        "interaction_type": str(plan.get("interaction_type", "ui_flow")),
        "ui_change_summary": str(plan.get("ui_change_summary", "")),
        "confidence": float(np.clip(float(plan.get("confidence", 0.0)), 0.0, 1.0)),
        "events": normalized_events,
        "composition_notes": str(plan.get("composition_notes", "")),
    }


def generate_haptic_plan_from_ui_flow() -> dict:
    client = ensure_openai_client()
    randomization_context = build_llm_randomization_context()
    prompt_text = LLM_USER_PROMPT
    if randomization_context["enabled"]:
        prompt_text = prompt_text + "\n\n" + randomization_context["instruction"]

    user_content = [{"type": "text", "text": prompt_text}]
    if LLM_RANDOM_VAE_INPUT_IGNORE_UI:
        user_content.append({
            "type": "text",
            "text": "No screenshots are attached because random VAE-input mode is enabled. Generate a random schema-valid VAE input plan from the catalog only.",
        })
    else:
        user_content.extend([
            {"type": "text", "text": "BEFORE screenshot:"},
            {"type": "image_url", "image_url": {"url": image_to_data_url(BEFORE_IMAGE_PATH, "BEFORE_IMAGE_PATH")}},
            {"type": "text", "text": "DURING screenshot:"},
            {"type": "image_url", "image_url": {"url": image_to_data_url(DURING_IMAGE_PATH, "DURING_IMAGE_PATH")}},
            {"type": "text", "text": "AFTER screenshot:"},
            {"type": "image_url", "image_url": {"url": image_to_data_url(AFTER_IMAGE_PATH, "AFTER_IMAGE_PATH")}},
        ])
    completion = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "system", "content": LLM_SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
        ],
        response_format=HAPTIC_PLAN_RESPONSE_FORMAT,
        temperature=LLM_TEMPERATURE,
        top_p=LLM_TOP_P,
    )
    raw_text = completion.choices[0].message.content
    parsed = json.loads(raw_text)
    normalized = normalize_event_plan(parsed)
    try:
        raw_openai_response = completion.model_dump(mode="json")
    except Exception:
        try:
            raw_openai_response = json.loads(completion.model_dump_json())
        except Exception:
            raw_openai_response = str(completion)
    return {
        "plan": normalized,
        "response_text": raw_text,
        "parsed_response": parsed,
        "raw_openai_response": raw_openai_response,
        "model": OPENAI_MODEL,
        "temperature": LLM_TEMPERATURE,
        "top_p": LLM_TOP_P,
        "randomization": randomization_context,
        "prompt_text": prompt_text,
        "catalog": PROTOTYPE_CATALOG_FOR_LLM,
        "catalog_text": PROTOTYPE_CATALOG_TEXT,
        "image_paths": {} if LLM_RANDOM_VAE_INPUT_IGNORE_UI else {
            "before": str(_coerce_path(BEFORE_IMAGE_PATH, "BEFORE_IMAGE_PATH")),
            "during": str(_coerce_path(DURING_IMAGE_PATH, "DURING_IMAGE_PATH")),
            "after": str(_coerce_path(AFTER_IMAGE_PATH, "AFTER_IMAGE_PATH")),
        },
    }


if USE_LLM_PLAN:
    LLM_PLAN_RESULT = generate_haptic_plan_from_ui_flow()
    HAPTIC_PLAN = LLM_PLAN_RESULT["plan"]
    PLAN_SOURCE = "llm"
else:
    HAPTIC_PLAN = normalize_event_plan(HAPTIC_PLAN)
    PLAN_SOURCE = "manual"

print("HAPTIC_PLAN:")
print(json.dumps(HAPTIC_PLAN, indent=2, ensure_ascii=False))


In [ ]:
# -----------------------------
# Event timeline mixer
# -----------------------------
def decode_z(z_values: list[float] | np.ndarray) -> np.ndarray:
    z = np.asarray(z_values, dtype=np.float32).reshape(1, -1)
    if z.shape[1] != latent_dim:
        raise ValueError(f"Expected latent_dim={latent_dim}, got z shape {z.shape}")
    z_t = torch.from_numpy(z).to(device)
    with torch.no_grad():
        decoded = model.decode(z_t, target_len=chunk_len)
    return decoded[0, 0].detach().cpu().numpy().astype(np.float32)


def fade_edges(x: np.ndarray, sr: int, fade_ms: float) -> np.ndarray:
    y = np.asarray(x, dtype=np.float32).copy()
    fade_n = int(round(sr * fade_ms / 1000.0))
    fade_n = max(0, min(fade_n, len(y) // 2))
    if fade_n > 1:
        ramp = np.linspace(0.0, 1.0, fade_n, dtype=np.float32)
        y[:fade_n] *= ramp
        y[-fade_n:] *= ramp[::-1]
    return y


def stretch_waveform(x: np.ndarray, target_len: int, mode: str = DEFAULT_DURATION_MODE) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32).reshape(-1)
    target_len = int(target_len)
    if target_len <= 0:
        raise ValueError("target_len must be positive")
    if target_len == len(x):
        return x.copy()
    if mode == "hold_stretch":
        source_idx = np.floor(np.arange(target_len) * len(x) / target_len).astype(np.int64)
        source_idx = np.clip(source_idx, 0, len(x) - 1)
        return x[source_idx].astype(np.float32)
    if mode == "linear_stretch":
        old_pos = np.linspace(0.0, 1.0, len(x), dtype=np.float32)
        new_pos = np.linspace(0.0, 1.0, target_len, dtype=np.float32)
        return np.interp(new_pos, old_pos, x).astype(np.float32)
    raise ValueError(f"Unsupported stretch mode: {mode}")


def prototype_from_event(event: dict) -> tuple[dict | None, list[str]]:
    behavior_class = event["behavior_class"]
    catalog = PROTOTYPE_CATALOG.get(behavior_class, [])
    warnings = []
    if not catalog:
        warnings.append(f"{event['event_id']}: no prototype available for {behavior_class}; using silence")
        return None, warnings
    requested_idx = int(event.get("prototype_index", 0))
    if requested_idx < 0 or requested_idx >= len(catalog):
        warnings.append(
            f"{event['event_id']}: prototype_index {requested_idx} out of range for "
            f"{behavior_class}; using 0"
        )
        requested_idx = 0
    proto = dict(catalog[requested_idx])
    proto["selection_index"] = requested_idx
    return proto, warnings


def decode_event(event: dict) -> tuple[dict, list[str]]:
    proto, warnings = prototype_from_event(event)
    duration_s = float(event["duration_s"])
    target_len = max(1, int(round(duration_s * sr)))
    strength = float(np.clip(float(event.get("strength", 0.6)), 0.0, 1.0))
    gain = 0.5 + 0.8 * strength

    if proto is None or not proto.get("z"):
        raw_chunk = np.zeros(chunk_len, dtype=np.float32)
        selected_raw_z = None
        selection_source = "none_silence"
    else:
        raw_chunk = decode_z(proto["z"])
        selected_raw_z = [float(v) for v in proto["z"]]
        selection_source = proto.get("selection_source")

    chunk = raw_chunk * gain
    chunk = stretch_waveform(chunk, target_len=target_len, mode=DEFAULT_DURATION_MODE)
    chunk = fade_edges(chunk, sr=sr, fade_ms=FADE_MS)
    chunk = np.clip(chunk, -clip_abs, clip_abs).astype(np.float32)
    record = {
        "event": dict(event),
        "selected_prototype": proto,
        "selected_raw_z": selected_raw_z,
        "selection_source": selection_source,
        "gain": float(gain),
        "duration_s": float(duration_s),
        "target_len": int(target_len),
        "chunk": chunk,
        "warnings": warnings,
    }
    return record, warnings


def compress_durations_to_fit(preferred: np.ndarray, gap_s: float) -> tuple[np.ndarray, float, dict]:
    durations = preferred.astype(np.float32).copy()
    n = len(durations)
    max_total_event_s = max(0.0, FINAL_DURATION_S - max(0, n - 1) * gap_s)
    summary = {
        "input_durations_s": preferred.astype(float).tolist(),
        "compressed": False,
        "compression_stage": "none",
        "gap_s": float(gap_s),
    }
    if float(np.sum(durations)) <= max_total_event_s + 1e-9:
        summary["output_durations_s"] = durations.astype(float).tolist()
        return durations, gap_s, summary

    flexible = durations > MIN_EVENT_DURATION_S
    if np.any(flexible):
        fixed_total = float(np.sum(np.where(flexible, 0.0, durations)))
        flexible_total = float(np.sum(durations[flexible]))
        target_flexible_total = max(0.0, max_total_event_s - fixed_total)
        if flexible_total > 0:
            scale = min(1.0, target_flexible_total / flexible_total)
            durations[flexible] = np.maximum(durations[flexible] * scale, MIN_EVENT_DURATION_S)
            summary.update({
                "compressed": True,
                "compression_stage": "long_full_first",
                "compression_scale": float(scale),
            })
    if float(np.sum(durations)) <= max_total_event_s + 1e-9:
        summary["output_durations_s"] = durations.astype(float).tolist()
        return durations, gap_s, summary

    if n > 1 and gap_s > MIN_EVENT_GAP_S:
        gap_s = MIN_EVENT_GAP_S
        max_total_event_s = max(0.0, FINAL_DURATION_S - (n - 1) * gap_s)
        summary.update({
            "compressed": True,
            "compression_stage": "reduced_gap",
            "gap_s": float(gap_s),
        })

    total = float(np.sum(durations))
    if total > max_total_event_s and total > 0:
        scale = max_total_event_s / total
        durations = np.maximum(durations * scale, 0.05)
        summary.update({
            "compressed": True,
            "compression_stage": "all_events_scaled",
            "compression_scale": float(scale),
        })

    summary["output_durations_s"] = durations.astype(float).tolist()
    return durations, gap_s, summary


def assign_event_timeline(events: list[dict]) -> tuple[list[dict], dict]:
    n = len(events)
    preferred = np.asarray([float(e["duration_s"]) for e in events], dtype=np.float32)
    durations, gap_s, compression = compress_durations_to_fit(preferred, EVENT_GAP_S)
    total_span_s = float(np.sum(durations) + max(0, n - 1) * gap_s)

    first_hint = events[0].get("timing_hint")
    last_event = events[-1]
    last_role = last_event.get("semantic_role")
    last_hint = last_event.get("timing_hint")
    end_aligned_roles = {"confirmation", "release", "notification", "warning"}

    if first_hint == "beginning":
        alignment = "beginning"
        sequence_start_s = 0.0
    elif last_hint == "end" or last_role in end_aligned_roles:
        alignment = "end_to_1p9s"
        sequence_start_s = max(0.0, TIMELINE_END_ALIGN_S - total_span_s)
    else:
        alignment = "center"
        sequence_start_s = max(0.0, (FINAL_DURATION_S - total_span_s) / 2.0)

    cursor_s = sequence_start_s
    scheduled = []
    for event, duration_s in zip(events, durations):
        start_idx = int(round(cursor_s * sr))
        event_len = max(1, int(round(float(duration_s) * sr)))
        end_idx = min(final_len, start_idx + event_len)
        if end_idx <= start_idx:
            end_idx = min(final_len, start_idx + 1)
        scheduled_event = dict(event)
        scheduled_event.update({
            "assigned_duration_s": float((end_idx - start_idx) / sr),
            "assigned_len": int(end_idx - start_idx),
            "start_s": float(start_idx / sr),
            "end_s": float(end_idx / sr),
            "start_idx": int(start_idx),
            "end_idx": int(end_idx),
        })
        scheduled.append(scheduled_event)
        cursor_s = (end_idx / sr) + gap_s

    timeline = {
        "event_gap_s": float(gap_s),
        "compression": compression,
        "alignment": alignment,
        "sequence_start_s": float(sequence_start_s),
        "total_span_s": float(total_span_s),
        "warnings": [],
    }
    if scheduled and scheduled[-1]["end_idx"] > final_len:
        timeline["warnings"].append("Timeline exceeded final_len and was clipped")
    return scheduled, timeline


def mix_event_chunks(scheduled_events: list[dict]) -> tuple[np.ndarray, list[dict], dict]:
    final = np.zeros(final_len, dtype=np.float32)
    records = []
    warnings = []
    for event in scheduled_events:
        record, event_warnings = decode_event(event)
        warnings.extend(event_warnings)
        target_len = int(event["assigned_len"])
        chunk = stretch_waveform(record["chunk"], target_len=target_len, mode=DEFAULT_DURATION_MODE)
        chunk = fade_edges(chunk, sr=sr, fade_ms=FADE_MS)
        start_idx = int(event["start_idx"])
        end_idx = int(event["end_idx"])
        available = max(0, min(len(chunk), final_len - start_idx, end_idx - start_idx))
        if available > 0:
            final[start_idx:start_idx + available] += chunk[:available]
        record.update({
            "chunk": chunk[:available].astype(np.float32),
            "start_idx": int(start_idx),
            "end_idx": int(start_idx + available),
            "start_s": float(start_idx / sr),
            "end_s": float((start_idx + available) / sr),
            "assigned_len": int(available),
            "assigned_duration_s": float(available / sr),
        })
        records.append(record)
    final = np.clip(final, -clip_abs, clip_abs).astype(np.float32)
    mixer = {
        "mode": "additive",
        "clip_abs": float(clip_abs),
        "fade_ms": float(FADE_MS),
        "duration_mode": DEFAULT_DURATION_MODE,
        "warnings": warnings,
    }
    return final, records, mixer


def event_record_for_json(record: dict) -> dict:
    item = {k: v for k, v in record.items() if k != "chunk"}
    if item.get("selected_prototype") is not None:
        item["prototype_index"] = item["selected_prototype"].get("prototype_index")
        item["sample_index"] = item["selected_prototype"].get("sample_index")
    else:
        item["prototype_index"] = None
        item["sample_index"] = None
    return item


HAPTIC_PLAN = normalize_event_plan(HAPTIC_PLAN)
scheduled_events, timeline_summary = assign_event_timeline(HAPTIC_PLAN["events"])
final_wave, event_records, mixer_summary = mix_event_chunks(scheduled_events)
timeline_summary["warnings"].extend(mixer_summary.get("warnings", []))
event_summaries = [event_record_for_json(record) for record in event_records]

print("Scheduled events:")
print(json.dumps(event_summaries, indent=2, ensure_ascii=False))
print("Timeline:")
print(json.dumps(timeline_summary, indent=2, ensure_ascii=False))
print(f"final max={np.max(np.abs(final_wave)):.4f}, std={np.std(final_wave):.4f}")


In [ ]:
# -----------------------------
# Save HapticGen-style validation artifacts plus per-event artifacts
# -----------------------------
SEQUENCE_SAMPLE_INTERVAL_MS = 10
SEQUENCE_SAMPLE_RATE_HZ = 1000 / SEQUENCE_SAMPLE_INTERVAL_MS
EXPECTED_SEQUENCE_LENGTH = int(round(FINAL_DURATION_S * SEQUENCE_SAMPLE_RATE_HZ))


def to_jsonable(value):
    if isinstance(value, dict):
        return {str(k): to_jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_jsonable(v) for v in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().tolist()
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(to_jsonable(payload), indent=2, ensure_ascii=False), encoding="utf-8")


def create_run_dir(base_dir: Path = PIPELINE_OUTPUT_DIR) -> Path:
    base_dir.mkdir(parents=True, exist_ok=True)
    run_dir = base_dir / datetime.now().strftime("%Y%m%d_%H%M%S")
    suffix = 1
    while run_dir.exists():
        run_dir = base_dir / f"{datetime.now().strftime('%Y%m%d_%H%M%S')}_{suffix:02d}"
        suffix += 1
    run_dir.mkdir(parents=True, exist_ok=False)
    return run_dir


def copy_input_images(run_dir: Path) -> dict[str, str]:
    images_dir = run_dir / "images"
    images_dir.mkdir(parents=True, exist_ok=True)
    if LLM_RANDOM_VAE_INPUT_IGNORE_UI:
        (images_dir / "README.txt").write_text(
            "No image inputs were used because random VAE-input mode was enabled.\n",
            encoding="utf-8",
        )
        return {}
    copied = {}
    for index, (label, source_value) in enumerate(
        [
            ("before", BEFORE_IMAGE_PATH),
            ("during", DURING_IMAGE_PATH),
            ("after", AFTER_IMAGE_PATH),
        ],
        start=1,
    ):
        source = _coerce_path(source_value, f"{label.upper()}_IMAGE_PATH").resolve()
        target = images_dir / f"image_{index:02d}_{label}{source.suffix.lower()}"
        shutil.copy2(source, target)
        copied[label] = str(target)
    return copied


def build_vae_text_input(plan: dict, events: list[dict]) -> str:
    lines = [
        f"Interaction type: {plan.get('interaction_type', 'ui_flow')}",
        f"Visual summary: {plan.get('ui_change_summary', '')}",
        f"Composition notes: {plan.get('composition_notes', '')}",
        "Ordered haptic event sequence:",
    ]
    for idx, record in enumerate(events, start=1):
        event = record.get("event", {})
        proto = record.get("selected_prototype") or {}
        lines.extend([
            f"{idx}. event_id={event.get('event_id')}",
            f"   event_type={event.get('event_type')}",
            f"   semantic_role={event.get('semantic_role')}",
            f"   style={event.get('style')}",
            f"   behavior_class={event.get('behavior_class')}",
            f"   strength={event.get('strength')}",
            f"   duration_hint={event.get('duration_hint')} assigned_duration_s={record.get('assigned_duration_s')}",
            f"   timing_hint={event.get('timing_hint')} start_s={record.get('start_s')} end_s={record.get('end_s')}",
            f"   prototype_index={proto.get('prototype_index')} sample_index={proto.get('sample_index')}",
            f"   reason={event.get('reason', '')}",
        ])
    return "\n".join(lines).strip() + "\n"


def save_event_waveform_plot(record: dict, output_path: Path) -> Path:
    chunk = np.asarray(record["chunk"], dtype=np.float32)
    t = np.arange(len(chunk)) / sr
    event = record["event"]
    plt.figure(figsize=(10, 2.6))
    plt.plot(t, chunk, linewidth=1.0)
    plt.title(
        f"{event['event_id']} | {event['semantic_role']} | "
        f"{event['behavior_class']} | proto={event.get('prototype_index')}"
    )
    plt.xlabel("time within event (s)")
    plt.ylabel("amplitude")
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close()
    return output_path


def save_pipeline_waveform_plot(output_path: Path) -> Path:
    time_final = np.arange(final_len) / sr
    fig, axes = plt.subplots(2, 1, figsize=(15, 7), gridspec_kw={"height_ratios": [1, 2]})
    for record in event_records:
        chunk = record["chunk"]
        t = np.arange(len(chunk)) / sr
        label = f"{record['event']['event_id']} {record['event']['behavior_class']}"
        axes[0].plot(t, chunk, linewidth=1.0, label=label)
    axes[0].set_title("Decoded event chunks after strength, stretch, and fade")
    axes[0].set_xlabel("time within event chunk (s)")
    axes[0].set_ylabel("amplitude")
    axes[0].legend(loc="upper right", fontsize=8)
    axes[0].grid(alpha=0.2)

    axes[1].plot(time_final, final_wave, linewidth=1.0, label="2s output")
    for record in event_records:
        event = record["event"]
        axes[1].axvspan(
            record["start_s"],
            record["end_s"],
            alpha=0.18,
            label=f"{event['event_id']} {event['semantic_role']}",
        )
    axes[1].set_title(f"Final 2s event timeline | {HAPTIC_PLAN.get('interaction_type')}")
    axes[1].set_xlabel("time (s)")
    axes[1].set_ylabel("amplitude")
    axes[1].legend(loc="upper right", fontsize=8)
    axes[1].grid(alpha=0.2)
    plt.tight_layout()
    fig.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return output_path


def save_arduino_sequence_json(output_path: Path, prompt: str, wav_path: Path | str, waveform_image_path: Path | str) -> Path:
    waveform = np.asarray(final_wave, dtype=np.float32).reshape(-1)
    samples_per_bin = max(1, int(round(sr * SEQUENCE_SAMPLE_INTERVAL_MS / 1000)))
    if len(waveform) == 0:
        envelope = np.asarray([], dtype=np.float32)
    elif len(waveform) < samples_per_bin:
        envelope = np.asarray([np.max(np.abs(waveform))], dtype=np.float32)
    else:
        usable_len = (len(waveform) // samples_per_bin) * samples_per_bin
        binned = waveform[:usable_len].reshape(-1, samples_per_bin)
        envelope = np.max(np.abs(binned), axis=1)
    peak = float(np.max(envelope)) if envelope.size else 0.0
    if peak > 0:
        sequence = np.round((envelope / peak) * 255).astype(np.uint8).tolist()
    else:
        sequence = [0 for _ in range(len(envelope))]
    if len(sequence) > EXPECTED_SEQUENCE_LENGTH:
        sequence = sequence[:EXPECTED_SEQUENCE_LENGTH]
    elif len(sequence) < EXPECTED_SEQUENCE_LENGTH:
        sequence = sequence + [0 for _ in range(EXPECTED_SEQUENCE_LENGTH - len(sequence))]
    sequence_duration_seconds = len(sequence) / SEQUENCE_SAMPLE_RATE_HZ
    payload = {
        "prompt": prompt,
        "generation_mode": "vae_decoded_waveform_envelope_uint8",
        "source_sample_rate": int(sr),
        "source_sample_count": int(len(waveform)),
        "target_duration_seconds": float(FINAL_DURATION_S),
        "sequence_sample_interval_ms": int(SEQUENCE_SAMPLE_INTERVAL_MS),
        "sequence_sample_rate_hz": float(SEQUENCE_SAMPLE_RATE_HZ),
        "samples_per_sequence_bin": int(samples_per_bin),
        "target_sequence_length": int(EXPECTED_SEQUENCE_LENGTH),
        "sequence_length": int(len(sequence)),
        "sequence_duration_seconds": float(sequence_duration_seconds),
        "sequence_uint8_csv": ",".join(str(value) for value in sequence),
        "sequence_uint8": sequence,
        "wav_path": str(wav_path),
        "waveform_image_path": str(waveform_image_path),
    }
    write_json(output_path, payload)
    return output_path


def create_artifact_bundle(run_dir: Path, artifact_paths: list[Path | str]) -> Path:
    bundle_path = run_dir / "download_all_artifacts.zip"
    seen = set()
    with zipfile.ZipFile(bundle_path, mode="w", compression=zipfile.ZIP_DEFLATED) as archive:
        for artifact_path in artifact_paths:
            path = Path(artifact_path)
            if path.exists() and path.resolve() not in seen:
                archive.write(path, arcname=path.name)
                seen.add(path.resolve())
        for folder_name in ("images", "events"):
            folder = run_dir / folder_name
            if folder.exists():
                for child in folder.iterdir():
                    if child.is_file() and child.resolve() not in seen:
                        archive.write(child, arcname=f"{folder_name}/{child.name}")
                        seen.add(child.resolve())
    return bundle_path


run_dir = create_run_dir()
copied_images = copy_input_images(run_dir)
events_dir = run_dir / "events"
events_dir.mkdir(parents=True, exist_ok=True)

wav_path = run_dir / "generated.wav"
npy_path = run_dir / "generated.npy"
preview_path = run_dir / "generated_waveform.png"
sequence_path = run_dir / "generated_sequence.json"
random_sequence_path = run_dir / RANDOM_SEQUENCE_JSON_NAME
metadata_path = run_dir / "generation_metadata.json"
request_path = run_dir / "three_image_request.json"
openai_response_path = run_dir / "openai_response.json"
vae_input_path = run_dir / "vae_input.txt"
plan_path = run_dir / "haptic_plan.json"

vae_text_input = build_vae_text_input(HAPTIC_PLAN, event_records)
vae_input_path.write_text(vae_text_input, encoding="utf-8")

np.save(npy_path, final_wave)
wav_write(wav_path, sr, final_wave.astype(np.float32))
save_pipeline_waveform_plot(preview_path)
save_arduino_sequence_json(sequence_path, prompt=vae_text_input, wav_path=wav_path, waveform_image_path=preview_path)
save_arduino_sequence_json(random_sequence_path, prompt=vae_text_input, wav_path=wav_path, waveform_image_path=preview_path)

for record in event_records:
    event_id = record["event"]["event_id"]
    event_wav_path = events_dir / f"{event_id}.wav"
    event_npy_path = events_dir / f"{event_id}.npy"
    event_plot_path = events_dir / f"{event_id}_waveform.png"
    event_metadata_path = events_dir / f"{event_id}_metadata.json"
    np.save(event_npy_path, record["chunk"])
    wav_write(event_wav_path, sr, np.asarray(record["chunk"], dtype=np.float32))
    save_event_waveform_plot(record, event_plot_path)
    record["artifacts"] = {
        "wav_path": str(event_wav_path),
        "npy_path": str(event_npy_path),
        "waveform_image_path": str(event_plot_path),
        "metadata_path": str(event_metadata_path),
    }
    write_json(event_metadata_path, event_record_for_json(record))

event_summaries = [event_record_for_json(record) for record in event_records]

request_payload = {
    "image_paths": copied_images,
    "original_image_paths": (LLM_PLAN_RESULT or {}).get("image_paths", {}),
    "openai_model": OPENAI_MODEL,
    "temperature": LLM_TEMPERATURE,
    "top_p": LLM_TOP_P,
    "randomization": (LLM_PLAN_RESULT or {}).get("randomization"),
    "system_prompt": LLM_SYSTEM_PROMPT,
    "user_prompt": (LLM_PLAN_RESULT or {}).get("prompt_text", LLM_USER_PROMPT),
    "response_format": HAPTIC_PLAN_RESPONSE_FORMAT,
    "prototype_catalog": PROTOTYPE_CATALOG_FOR_LLM,
}
write_json(request_path, request_payload)

openai_response_payload = {
    "visual_summary": HAPTIC_PLAN.get("ui_change_summary", ""),
    "haptic_plan": HAPTIC_PLAN,
    "vae_text_input": vae_text_input,
    "catalog_choice": [
        {
            "event_id": record["event"]["event_id"],
            "behavior_class": record["event"]["behavior_class"],
            "prototype_index": (record.get("selected_prototype") or {}).get("prototype_index"),
            "sample_index": (record.get("selected_prototype") or {}).get("sample_index"),
            "semantic_role": record["event"]["semantic_role"],
            "style": record["event"]["style"],
        }
        for record in event_records
    ],
    "response_text": (LLM_PLAN_RESULT or {}).get("response_text"),
    "parsed_response": (LLM_PLAN_RESULT or {}).get("parsed_response"),
    "raw_openai_response": (LLM_PLAN_RESULT or {}).get("raw_openai_response"),
    "model": (LLM_PLAN_RESULT or {}).get("model", OPENAI_MODEL),
    "temperature": (LLM_PLAN_RESULT or {}).get("temperature", LLM_TEMPERATURE),
    "top_p": (LLM_PLAN_RESULT or {}).get("top_p", LLM_TOP_P),
    "randomization": (LLM_PLAN_RESULT or {}).get("randomization"),
}
write_json(openai_response_path, openai_response_payload)

saved_plan = {
    "haptic_plan": HAPTIC_PLAN,
    "plan_source": PLAN_SOURCE,
    "vae_text_input": vae_text_input,
    "events": event_summaries,
    "timeline": timeline_summary,
    "mixer": mixer_summary,
}
write_json(plan_path, saved_plan)

final_nonzero = np.flatnonzero(np.abs(final_wave) > 1e-7)
metadata = {
    "pipeline_name": "ui_flow_to_vae_haptic_pipeline",
    "pipeline_version": "v3_event_timeline_mixer",
    "created_at": datetime.now().isoformat(),
    "run_dir": str(run_dir),
    "project_root": str(PROJECT_ROOT),
    "plan_source": PLAN_SOURCE,
    "openai": {
        "model": OPENAI_MODEL,
        "temperature": LLM_TEMPERATURE,
        "top_p": LLM_TOP_P,
        "randomized_output": bool(LLM_RANDOMIZE_OUTPUT),
        "random_vae_input_ignore_ui": bool(LLM_RANDOM_VAE_INPUT_IGNORE_UI),
        "randomization": (LLM_PLAN_RESULT or {}).get("randomization"),
        "response_json_path": str(openai_response_path),
        "request_json_path": str(request_path),
    },
    "vae_input": {
        "text_input_path": str(vae_input_path),
        "text_input": vae_text_input,
        "note": "The VAE decoder receives each selected raw z; this text is the semantic event sequence used to choose behavior prototypes.",
    },
    "vae_parameters": {
        "config_path": str(PROJECT_ROOT / CONFIG_PATH),
        "checkpoint_path": str(checkpoint_path),
        "device": str(device),
        "latent_dim": int(latent_dim),
        "sample_rate": int(sr),
        "vae_chunk_len": int(chunk_len),
        "final_duration_s": float(FINAL_DURATION_S),
        "final_len": int(final_len),
        "clip_abs": float(clip_abs),
        "fade_ms": float(FADE_MS),
        "strength_gain_formula": "gain = 0.5 + 0.8 * strength",
        "duration_mode": DEFAULT_DURATION_MODE,
        "config": config,
    },
    "events": event_summaries,
    "timeline": timeline_summary,
    "mixer": mixer_summary,
    "semantic_to_z_learning": {
        "current_method": "semantic -> class/prototype choice -> raw z",
        "future_hooks": [
            "event.semantic_role",
            "event.style",
            "event.strength",
            "event.duration_hint",
            "event.timing_hint",
            "selected_raw_z",
            "decoded_waveform_features",
        ],
    },
    "waveform_stats": {
        "max_abs": float(np.max(np.abs(final_wave))) if len(final_wave) else 0.0,
        "mean": float(np.mean(final_wave)) if len(final_wave) else 0.0,
        "std": float(np.std(final_wave)) if len(final_wave) else 0.0,
        "nonzero_start_idx": int(final_nonzero[0]) if final_nonzero.size else None,
        "nonzero_end_idx": int(final_nonzero[-1] + 1) if final_nonzero.size else None,
    },
    "artifacts": {
        "images_dir": str(run_dir / "images"),
        "events_dir": str(events_dir),
        "three_image_request_json": str(request_path),
        "openai_response_json": str(openai_response_path),
        "vae_input_txt": str(vae_input_path),
        "haptic_plan_json": str(plan_path),
        "wav_path": str(wav_path),
        "npy_path": str(npy_path),
        "waveform_image_path": str(preview_path),
        "sequence_json_path": str(sequence_path),
        "metadata_path": str(metadata_path),
    },
    "atlas": {
        "curated_json_path": str(CURATED_JSON_PATH),
        "atlas_json_path": str(ATLAS_JSON_PATH),
        "latent_table_path": str(LATENT_TABLE_PATH),
    },
}
write_json(metadata_path, metadata)

bundle_path = create_artifact_bundle(
    run_dir,
    [
        request_path,
        openai_response_path,
        vae_input_path,
        plan_path,
        wav_path,
        npy_path,
        preview_path,
        sequence_path,
        metadata_path,
    ],
)
metadata["artifacts"]["download_all_artifacts_zip"] = str(bundle_path)
write_json(metadata_path, metadata)

print(f"Saved output directory: {run_dir}")
print(f"REQUEST:  {request_path}")
print(f"OPENAI:   {openai_response_path}")
print(f"VAE TXT:  {vae_input_path}")
print(f"PLAN:     {plan_path}")
print(f"WAV:      {wav_path}")
print(f"NPY:      {npy_path}")
print(f"PLOT:     {preview_path}")
print(f"SEQ:      {sequence_path}")
print(f"RANDSEQ:  {random_sequence_path}")
print(f"META:     {metadata_path}")
print(f"EVENTS:   {events_dir}")
print(f"ZIP:      {bundle_path}")


In [ ]:
# -----------------------------
# Plot and playback
# -----------------------------
time_final = np.arange(final_len) / sr

fig, axes = plt.subplots(2, 1, figsize=(15, 7), gridspec_kw={"height_ratios": [1, 2]})

for record in event_records:
    chunk = record["chunk"]
    t = np.arange(len(chunk)) / sr
    event = record["event"]
    axes[0].plot(t, chunk, linewidth=1.0, label=f"{event['event_id']} {event['behavior_class']}")
axes[0].set_title("Decoded event chunks")
axes[0].set_xlabel("time within event chunk (s)")
axes[0].set_ylabel("amplitude")
axes[0].legend(loc="upper right", fontsize=8)
axes[0].grid(alpha=0.2)

axes[1].plot(time_final, final_wave, linewidth=1.0, label="2s output")
for record in event_records:
    event = record["event"]
    axes[1].axvspan(
        record["start_s"],
        record["end_s"],
        alpha=0.18,
        label=f"{event['event_id']} {event['semantic_role']}",
    )
axes[1].set_title(f"Final 2s waveform | {HAPTIC_PLAN.get('interaction_type')}")
axes[1].set_xlabel("time (s)")
axes[1].set_ylabel("amplitude")
axes[1].legend(loc="upper right", fontsize=8)
axes[1].grid(alpha=0.2)

plt.tight_layout()
plt.show()

print(f"Saved preview: {preview_path}")
display(Audio(final_wave, rate=sr))


In [ ]:
# -----------------------------
# Smoke assertions for v3 event timeline mixer
# -----------------------------
assert len(final_wave) == 16000, f"Expected 16000 samples for 2s at 8kHz, got {len(final_wave)}"
assert chunk_len == 4000, f"Expected original 4000-sample VAE chunk, got {chunk_len}"
assert 1 <= len(event_records) <= MAX_EVENTS, len(event_records)

for record in event_records:
    assert 0 <= record["start_idx"] < record["end_idx"] <= final_len, record
    event_id = record["event"]["event_id"]
    artifacts = record.get("artifacts", {})
    for key in ("wav_path", "npy_path", "waveform_image_path", "metadata_path"):
        assert key in artifacts and Path(artifacts[key]).exists(), (event_id, key, artifacts)

outside = final_wave.copy()
for record in event_records:
    outside[record["start_idx"]:record["end_idx"]] = 0.0
assert np.max(np.abs(outside)) < 1e-6, "Samples outside assigned event regions should remain silent"

sequence_payload = load_json(sequence_path)
random_sequence_payload = load_json(random_sequence_path)
assert sequence_payload["sequence_length"] == 200, sequence_payload["sequence_length"]
assert random_sequence_payload["sequence_length"] == 200, random_sequence_payload["sequence_length"]
assert sequence_payload["sequence_sample_interval_ms"] == 10
assert random_sequence_payload["sequence_sample_interval_ms"] == 10
assert len(sequence_payload["sequence_uint8"]) == 200
assert len(random_sequence_payload["sequence_uint8"]) == 200
assert all(isinstance(v, int) and 0 <= v <= 255 for v in sequence_payload["sequence_uint8"]), "sequence_uint8 must contain 200 uint8 values"
assert all(isinstance(v, int) and 0 <= v <= 255 for v in random_sequence_payload["sequence_uint8"]), "sequence_uint8 must contain 200 uint8 values"
assert sequence_payload["generation_mode"] == "vae_decoded_waveform_envelope_uint8"
assert random_sequence_payload["generation_mode"] == "vae_decoded_waveform_envelope_uint8"

required_top_level = [
    run_dir / "images",
    run_dir / "events",
    run_dir / "three_image_request.json",
    run_dir / "openai_response.json",
    run_dir / "vae_input.txt",
    run_dir / "haptic_plan.json",
    run_dir / "generated.wav",
    run_dir / "generated.npy",
    run_dir / "generated_waveform.png",
    run_dir / "generated_sequence.json",
    run_dir / RANDOM_SEQUENCE_JSON_NAME,
    run_dir / "generation_metadata.json",
    run_dir / "download_all_artifacts.zip",
]
for path in required_top_level:
    assert path.exists(), path

print("v3 smoke checks passed.")
print(f"events: {len(event_records)}")
for record in event_records:
    event = record["event"]
    print(
        f"{event['event_id']}: {event['behavior_class']} {event['semantic_role']} "
        f"start={record['start_s']:.3f}s end={record['end_s']:.3f}s "
        f"sample_index={(record.get('selected_prototype') or {}).get('sample_index')}"
    )
